# Faulty parcels in 200 checked: np chart, month 2

**np Control Chart** · a worked example from the StatNex teaching guide

**The story.** A warehouse checks 200 parcels every day and counts how many are damaged or contain the wrong item. These are its counts for a second month. In the month before, every day stayed inside the control limits. This month, a new temporary worker joined the packing line on day 18.

**The question.** Is the error count stable from day to day this month, or are there days that need investigating?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/np_chart__month2.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Day,Parcels checked,Damaged or wrong
1.0,200.0,4.0
2.0,200.0,7.0
3.0,200.0,5.0
4.0,200.0,5.0
5.0,200.0,5.0
6.0,200.0,6.0
7.0,200.0,4.0
8.0,200.0,2.0
9.0,200.0,1.0
10.0,200.0,2.0
11.0,200.0,5.0
12.0,200.0,1.0
13.0,200.0,8.0
14.0,200.0,2.0
15.0,200.0,5.0
16.0,200.0,3.0
17.0,200.0,2.0
18.0,200.0,13.0
19.0,200.0,9.0
20.0,200.0,4.0
21.0,200.0,5.0
22.0,200.0,4.0
23.0,200.0,2.0
24.0,200.0,3.0
25.0,200.0,1.0

These cells save `control_signals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile control_signals.py
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from control_signals import draw, pattern_signals

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per subgroup: the number of nonconforming (faulty) items found when the same
# number of items, n, was inspected every time. Blanks, text and negative counts stop
# the chart.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Damaged or wrong"
n = 200  # items inspected per subgroup
defectives = pd.to_numeric(data[column], errors="coerce")
if defectives.isna().any() or (defectives < 0).any() or (defectives > n).any():
    raise ValueError("Every count must be a number from 0 to the subgroup size.")
defectives.index = np.arange(1, len(defectives) + 1)  # subgroup numbers 1, 2, ...


def chart(counts):
    """Binomial limits: centre n p-bar, 3 sigma either side, no lower limit below 0."""
    # Each count is treated as binomial: n items, each faulty with the same chance p,
    # independently of the others. p is estimated by p-bar, all faulty items over all
    # items inspected, and the limits sit 3 binomial standard deviations from the centre:
    #   CL = n p-bar,  UCL, LCL = n p-bar +/- 3 sqrt(n p-bar (1 - p-bar))
    # A lower limit below 0 is drawn at 0, so the chart then cannot flag a good subgroup.
    p_bar = counts.sum() / (n * len(counts))
    centre = n * p_bar
    margin = 3.0 * np.sqrt(centre * (1 - p_bar))
    return pd.DataFrame({"np": counts, "CL": centre, "UCL": centre + margin,
                         "LCL": max(centre - margin, 0.0)})


def outside(table):
    return (table["np"] > table["UCL"]) | (table["np"] < table["LCL"])


counts = defectives

## Control chart data

In [ ]:
# One row per subgroup: its count of faulty items against the centre line (CL) and the
# limits (UCL, LCL). A count outside the limits is a signal that something other than
# everyday variation affected that subgroup. "In control" means stable and predictable,
# not good enough: whether the average rate is acceptable is a separate business target.
table = chart(counts)
print()
print(table)
print(f"\nSubgroups outside the limits: {list(table.index[outside(table)])}")

## The chart

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row.
for message in pattern_signals("np chart", table["np"], table["CL"]):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/np_chart.png", "np chart", table["np"], table["CL"], table["UCL"], table["LCL"])

## What the result says

This chart carries the warehouse story into a second month. An np chart follows the number of faulty items in samples of a fixed size, here 200 parcels a day. The centre line (CL), the average, is 4.32 faulty parcels a day, almost the same as the month before. The upper control limit (UCL), the highest count that ordinary chance variation gives, is 10.49.

Day 18 has 13 faulty parcels, which is above the UCL. That is the day the new temporary worker joined the packing line. Day 19 has 9, high but inside the limits, and every other day is ordinary.

Last month's worst day was noise. This month's worst day is a signal: day 18 is the one to investigate.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.